# Init

Create agnostic Bridges between the dilated and constricted states.

In [ ]:
import copy
import importlib
import os
import pickle
import sys
import xml.etree.cElementTree as ET

import matplotlib.pyplot as plt
import mdtraj
# https://github.com/mellofariam/mmct
import mmct.mmct.force_field as ff
import mmct.mmct.macromolecule as mol
import mmct.mmct.pdb_tools as pdb_tools
import numpy as np
import pandas as pd
from scipy.spatial import distance

sys.path.append("../../")
from npc_ana_tool import *

In [2]:
# work_dir = '/home/ed31/Documents/LargeSystem/npc'
work_dir = '../../'

# Load data

In [3]:
df_c, top_c, xml_c = ff.load_force_field(
    pdb_file=f"{work_dir}/emin/constricted/reconstruct/constricted_reconstruct.pdb",
    top_file=f"{work_dir}/SBM/dual_basin/predual_constricted_CA.top",
    xml_file=f"{work_dir}/SBM/dual_basin/predual_constricted_CA.xml",
)

In [4]:
df_d, top_d, xml_d = ff.load_force_field(
    pdb_file=f"{work_dir}/emin/dilated/reconstruct/dilated_reconstruct.pdb",
    top_file=f"{work_dir}/SBM/dual_basin/predual_dilated_CA.top",
    xml_file=f"{work_dir}/SBM/dual_basin/predual_dilated_CA.xml",
)

In [6]:
# Rename the chain ids

count = 0
for i in range(len(df_chain_info)):
    start = df_chain_info.iloc[i]['CA_start']
    end = df_chain_info.iloc[i]['CA_end']
    for j in range(8):
        sub_start = start + j * N_subunit_CA 
        sub_end = end + j * N_subunit_CA
        df_c.loc[sub_start:sub_end+1, 'chain_id'] =  str(count)
        df_d.loc[sub_start:sub_end+1, 'chain_id'] =  str(count)
        count += 1

In [7]:
# Set up the trivial index mapping

dict_idx = dict()
for i in range(1, 1+N_subunit_CA*8):
    dict_idx[i] = i

In [8]:
# 'CR_all' 'IR_all' 'NR_all' 'LR_all' 'Bridge'
ring_indices = get_ring_indices(df_chain_info)
ring_all_indices = get_ring_allsubunits_indices(ring_indices)

# Functions

The functions are modifed from the `mmct` package, aimed at processing part of the system instead of the entire one.

In [9]:
def process_bonds(
    reference_top: dict[str, pd.DataFrame],
    additional_top: dict[str, pd.DataFrame],
    multibasin_top: dict[str, pd.DataFrame],
    idx_from_additional_to_reference: dict[int, int],
    multibasin_xml: ET.ElementTree,
    atom_set: set[int],
):
    """
    Processes bonds from the reference and additional topologies,
    converting them to the reference indices and merging them.
    The final bonds are averages of the equilibrium length in both
    structures.
    """

    bonds_converted_to_reference = {
        col: [] for col in additional_top["bonds"].columns
    }

    for _, row in additional_top["bonds"].iterrows():

        i = idx_from_additional_to_reference.get(row["ai"])
        j = idx_from_additional_to_reference.get(row["aj"])

        if i is not None and j is not None:
            bonds_converted_to_reference["ai"].append(i)
            bonds_converted_to_reference["aj"].append(j)
            bonds_converted_to_reference["func"].append(row["func"])
            if "r0(nm)" in row and "Kb" in row:
                bonds_converted_to_reference["r0(nm)"].append(row["r0(nm)"])
                bonds_converted_to_reference["Kb"].append(row["Kb"])

    df_bonds_additional = pd.DataFrame(
        data=bonds_converted_to_reference
    )
    df_bonds_additional[["ai", "aj", "func"]] = df_bonds_additional[
        ["ai", "aj", "func"]
    ].astype(int)
    if ("r0(nm)" in df_bonds_additional.columns) and ("Kb" in df_bonds_additional.columns):
        df_bonds_additional[["r0(nm)", "Kb"]] = \
        df_bonds_additional[["r0(nm)", "Kb"]].astype(float)

    merged_bonds = pd.merge(
        reference_top["bonds"],
        df_bonds_additional,
        left_on=["ai", "aj", "func"],
        right_on=["ai", "aj", "func"],
        how="outer",
        suffixes=("_1", "_2"),
        indicator="source",
    )

    if "Kb" in df_bonds_additional.columns:
        merged_bonds["Kb"] = np.nanmax(
            merged_bonds[["Kb_1", "Kb_2"]],
            axis=1,
        )

    if "r0(nm)" in df_bonds_additional.columns:
        merged_bonds.loc[
            merged_bonds["source"] == "left_only", "r0(nm)_2"
        ] =\
        merged_bonds.loc[
            merged_bonds["source"] == "left_only", "r0(nm)_1"
        ]

        merged_bonds.loc[
            merged_bonds["source"] == "right_only", "r0(nm)_1"
        ] =\
        merged_bonds.loc[
            merged_bonds["source"] == "right_only", "r0(nm)_2"
        ]

        # Only change the bonds that are in the atom_set
        merged_bonds["r0(nm)"] = np.nan

        merged_bonds.loc[
            (merged_bonds['ai'] - 1).isin(atom_set), "r0(nm)"
        ] =\
        np.nanmean(
            merged_bonds.loc[
                (merged_bonds['ai'] - 1).isin(atom_set), ["r0(nm)_1", "r0(nm)_2"]
            ],
            axis=1,
        )

        merged_bonds.loc[
            ~((merged_bonds['ai'] - 1).isin(atom_set)), "r0(nm)"
        ] =\
        merged_bonds.loc[
            ~((merged_bonds['ai'] - 1).isin(atom_set)), "r0(nm)_1"
        ]

    multibasin_top["bonds"] = merged_bonds[
        list(df_bonds_additional.columns)
    ].reset_index(drop=True)

    if "bondtypes" in reference_top or "bondtypes" in additional_top:
        multibasin_top["bondtypes"] = pd.merge(
            left=reference_top.get("bondtypes", pd.DataFrame()),
            right=additional_top.get("bondtypes", pd.DataFrame()),
            how="inner",
            indicator=False,
        ).reset_index(drop=True)

    return multibasin_top, multibasin_xml

In [10]:
def process_angles(
    reference_top: dict[str, pd.DataFrame],
    additional_top: dict[str, pd.DataFrame],
    multibasin_top: dict[str, pd.DataFrame],
    idx_from_additional_to_reference: dict[int, int],
    multibasin_xml: ET.ElementTree,
    atom_set: set[int],
    # mode: str = "middle",
) -> tuple[dict[str, pd.DataFrame], ET.ElementTree]:
    """
    Processes angles from the reference and additional topologies,
    converting them to the reference indices and merging them.
    The angles are either averaged or converted to a flat-bottom potential.
    """

    angles_converted_to_reference = {
        col: [] for col in additional_top["angles"].columns
    }

    multibasin_root = multibasin_xml.getroot()
    if multibasin_root is None:
        raise ValueError("The XML tree is empty or malformed.")

    for _, row in additional_top["angles"].iterrows():

        i = idx_from_additional_to_reference.get(row["ai"])
        j = idx_from_additional_to_reference.get(row["aj"])
        k = idx_from_additional_to_reference.get(row["ak"])

        if i is not None and j is not None and k is not None:
            angles_converted_to_reference["ai"].append(i)
            angles_converted_to_reference["aj"].append(j)
            angles_converted_to_reference["ak"].append(k)
            angles_converted_to_reference["func"].append(row["func"])

            if "th0(deg)" in row and "Ka" in row:
                angles_converted_to_reference["th0(deg)"].append(
                    row["th0(deg)"]
                )
                angles_converted_to_reference["Ka"].append(row["Ka"])

    df_angles_additional = pd.DataFrame(
        data=angles_converted_to_reference
    )
    df_angles_additional[["ai", "aj", "ak", "func"]] = (
        df_angles_additional[["ai", "aj", "ak", "func"]].astype(int)
    )
    if (
        "th0(deg)" in df_angles_additional.columns
        and "Ka" in df_angles_additional.columns
    ):
        df_angles_additional[["th0(deg)", "Ka"]] = (
            df_angles_additional[["th0(deg)", "Ka"]].astype(float)
        )

    if "angles" in reference_top:
        df_angles_reference = reference_top["angles"].copy()
    else:
        # angles information present in XML
        angles_reference = {
            "ai": [],
            "aj": [],
            "ak": [],
            "func": [],
            "theta1": [],
            "theta2": [],
            "Ka": [],
        }
        for angle_type in multibasin_root.findall(
            ".//angles/angles_type"
        ):
            for interaction in angle_type.findall("interaction"):
                i = int(interaction.attrib["i"])
                j = int(interaction.attrib["j"])
                k = int(interaction.attrib["k"])

                angles_reference["ai"].append(i)
                angles_reference["aj"].append(j)
                angles_reference["ak"].append(k)
                angles_reference["func"].append(1)
                angles_reference["theta1"].append(
                    np.degrees(float(interaction.attrib["theta1"]))
                )
                angles_reference["theta2"].append(
                    np.degrees(float(interaction.attrib["theta2"]))
                )
                angles_reference["Ka"].append(
                    float(interaction.attrib["Ka"])
                )
        df_angles_reference = pd.DataFrame(data=angles_reference)
        df_angles_reference[["ai", "aj", "ak", "func"]] = (
            df_angles_reference[["ai", "aj", "ak", "func"]].astype(
                int
            )
        )
        df_angles_reference[["theta1", "theta2", "Ka"]] = (
            df_angles_reference[["theta1", "theta2", "Ka"]].astype(
                float
            )
        )
        df_angles_reference["th0(deg)"] = np.nanmean(
            df_angles_reference[["theta1", "theta2"]],
            axis=1,
        )
        df_angles_reference["editable"] = True
        df_angles_reference.loc[
            df_angles_reference["theta1"]
            != df_angles_reference["theta2"],
            "editable",
        ] = False

        df_angles_reference.drop(
            ["theta1", "theta2"], axis=1, inplace=True
        )

    merged_angles = pd.merge(
        df_angles_reference,
        df_angles_additional,
        left_on=["ai", "aj", "ak", "func"],
        right_on=["ai", "aj", "ak", "func"],
        how="outer",
        suffixes=("_1", "_2"),
        indicator="source",
    )

    if "Ka" in df_angles_additional.columns:
        merged_angles["Ka"] = np.nanmax(
            merged_angles[["Ka_1", "Ka_2"]],
            axis=1,
        )

    if "editable" in df_angles_reference.columns:
        # check if any of the angles in both structures are not editable
        if len(
            merged_angles.loc[
                (merged_angles["source"] == "both")
                & (merged_angles["editable"] == False)
            ]
        ):
            raise ValueError(
                "Angles already edited before is trying to be edited again. "
                "This is not supported. "
            )

    if "th0(deg)" in df_angles_additional.columns:
        merged_angles.loc[
            merged_angles["source"] == "left_only", "th0(deg)_2"
        ] = merged_angles.loc[
            merged_angles["source"] == "left_only", "th0(deg)_1"
        ]
        merged_angles.loc[
            merged_angles["source"] == "right_only", "th0(deg)_1"
        ] = merged_angles.loc[
            merged_angles["source"] == "right_only", "th0(deg)_2"
        ]

    # Only implement a flat bottom potentail
    # remove angles from multibasin_xml if they exist
    try:
        multibasin_angles_root = multibasin_root.find("angles")
        multibasin_angles_root[:] = [] # Clear the existing contacts
    except:
        multibasin_angles_root = ET.SubElement(multibasin_root, "angles")

    # Harmonic potential for atoms not in atom_set
    harmonic_xml = ET.SubElement(
        multibasin_angles_root, 
        "angles_type",
        attrib={"name": "angle_harmonic"}
    )
    ET.SubElement(
        harmonic_xml,
        "expression",
        attrib={
            "expr": "Ka / 2 * ((theta0 - theta)^2)"
        },
    )
    ET.SubElement(harmonic_xml, "parameter").text = "Ka"
    ET.SubElement(harmonic_xml, "parameter").text = "theta0"

    # Flat-bottom potential for atoms in atom_set
    flat_bottom_xml = ET.SubElement(
        multibasin_angles_root,
        "angles_type",
        attrib={"name": "angle_flat_bottom"},
    )
    ET.SubElement(
        flat_bottom_xml,
        "expression",
        attrib={
            "expr": "Ka / 2 * ((theta1 - theta)^2 * step(theta1 - theta) + (theta - theta2)^2 * step(theta-theta2))"
        },
    )
    ET.SubElement(flat_bottom_xml, "parameter").text = "Ka"
    ET.SubElement(flat_bottom_xml, "parameter").text = "theta1"
    ET.SubElement(flat_bottom_xml, "parameter").text = "theta2"

    merged_angles.rename(columns={"th0(deg)_1": "th0_deg_1", "th0(deg)_2": "th0_deg_2"}, inplace=True) # inplace to save memory
    for row in merged_angles.itertuples():
        # Only change the angles that are in the atom_set
        if (row.ai - 1) in atom_set:
            theta1 = min(row.th0_deg_1, row.th0_deg_2)
            theta2 = max(row.th0_deg_1, row.th0_deg_2)
            ET.SubElement(
                flat_bottom_xml,
                "interaction",
                attrib={
                    "i": str(row.ai),
                    "j": str(row.aj),
                    "k": str(row.ak),
                    "Ka": f"{row.Ka:.5e}",
                    "theta1": f"{np.radians(theta1):.5e}",
                    "theta2": f"{np.radians(theta2):.5e}",
                },
            )
        else:
            theta0 = row.th0_deg_1
            ET.SubElement(
                harmonic_xml,
                "interaction",
                attrib={
                    "i": str(row.ai),
                    "j": str(row.aj),
                    "k": str(row.ak),
                    "Ka": f"{row.Ka:.5e}",
                    "theta0": f"{np.radians(theta0):.5e}",
                },
            )
    merged_angles.rename(columns={"th0_deg_1": "th0(deg)_1", "th0_deg_2": "th0(deg)_2"}, inplace=True)

    if "angles" in multibasin_top:
        del multibasin_top["angles"]

    return multibasin_top, multibasin_xml


In [11]:
def process_dihedrals(
    reference_xml: ET.ElementTree,
    additional_xml: ET.ElementTree,
    multibasin_xml: ET.ElementTree,
    idx_from_additional_to_reference: dict[int, int],
    atom_set: set[int],
) -> ET.ElementTree:
    """
    Processes dihedrals from the reference and additional XML files,
    converting them to the reference indices and merging them.
    The final dihedrals are averages of the parameters in both structures.
    """

    reference_root = reference_xml.getroot()
    if reference_root is None:
        raise ValueError(
            "The reference XML tree is empty or malformed."
        )
    additional_root = additional_xml.getroot()
    if additional_root is None:
        raise ValueError(
            "The additional XML tree is empty or malformed."
        )
    multibasin_root = multibasin_xml.getroot()
    if multibasin_root is None:
        raise ValueError(
            "The multibasin XML tree is empty or malformed."
        )

    dihedrals_converted_to_reference = {}
    df_additional = {}

    for dihedral_type in additional_root.findall(
        ".//dihedrals/dihedrals_type"
    ):
        dihedral_type_name = dihedral_type.get("name")

        dihedrals_converted_to_reference[dihedral_type_name] = {
            col: []
            for col in dihedral_type.findall("interaction")[
                0
            ].attrib.keys()
        }

        no_indices_columns = []
        for col in dihedrals_converted_to_reference[
            dihedral_type_name
        ]:
            if col not in ["i", "j", "k", "l"]:
                no_indices_columns.append(col)

        for interaction in dihedral_type.findall("interaction"):
            # Convert dihedral indices from additional to reference
            i = idx_from_additional_to_reference.get(
                int(interaction.attrib["i"])
            )
            j = idx_from_additional_to_reference.get(
                int(interaction.attrib["j"])
            )
            k = idx_from_additional_to_reference.get(
                int(interaction.attrib["k"])
            )
            l = idx_from_additional_to_reference.get(
                int(interaction.attrib["l"])
            )

            if (
                i is not None
                and j is not None
                and k is not None
                and l is not None
            ):
                dihedrals_converted_to_reference[dihedral_type_name][
                    "i"
                ].append(i)
                dihedrals_converted_to_reference[dihedral_type_name][
                    "j"
                ].append(j)
                dihedrals_converted_to_reference[dihedral_type_name][
                    "k"
                ].append(k)
                dihedrals_converted_to_reference[dihedral_type_name][
                    "l"
                ].append(l)

                for key in no_indices_columns:
                    dihedrals_converted_to_reference[
                        dihedral_type_name
                    ][key].append(interaction.attrib[key])

        df_additional[dihedral_type_name] = pd.DataFrame(
            data=dihedrals_converted_to_reference[dihedral_type_name]
        )
        df_additional[dihedral_type_name][["theta0", "weight"]] = (
            df_additional[dihedral_type_name][
                ["theta0", "weight"]
            ].astype(float)
        )
        df_additional[dihedral_type_name][["i", "j", "k", "l"]] = (
            df_additional[dihedral_type_name][
                ["i", "j", "k", "l"]
            ].astype(int)
        )
        if (
            "multiplicity"
            in df_additional[dihedral_type_name].columns
        ):
            df_additional[dihedral_type_name][["multiplicity"]] = (
                df_additional[dihedral_type_name][
                    ["multiplicity"]
                ].astype(int)
            )

    dihedrals_in_reference = {}
    df_reference = {}

    for dihedral_type in reference_root.findall(
        ".//dihedrals/dihedrals_type"
    ):
        dihedral_type_name = dihedral_type.get("name")

        dihedrals_in_reference[dihedral_type_name] = {
            col: []
            for col in dihedral_type.findall("interaction")[
                0
            ].attrib.keys()
        }

        no_indices_columns = []
        for col in dihedrals_in_reference[dihedral_type_name]:
            if col not in ["i", "j", "k", "l"]:
                no_indices_columns.append(col)

        for interaction in dihedral_type.findall("interaction"):
            i = int(interaction.attrib["i"])
            j = int(interaction.attrib["j"])
            k = int(interaction.attrib["k"])
            l = int(interaction.attrib["l"])

            dihedrals_in_reference[dihedral_type_name]["i"].append(i)
            dihedrals_in_reference[dihedral_type_name]["j"].append(j)
            dihedrals_in_reference[dihedral_type_name]["k"].append(k)
            dihedrals_in_reference[dihedral_type_name]["l"].append(l)

            for key in no_indices_columns:
                dihedrals_in_reference[dihedral_type_name][
                    key
                ].append(interaction.attrib[key])

        df_reference[dihedral_type_name] = pd.DataFrame(
            data=dihedrals_in_reference[dihedral_type_name]
        )
        df_reference[dihedral_type_name][["theta0", "weight"]] = (
            df_reference[dihedral_type_name][
                ["theta0", "weight"]
            ].astype(float)
        )
        df_reference[dihedral_type_name][["i", "j", "k", "l"]] = (
            df_reference[dihedral_type_name][
                ["i", "j", "k", "l"]
            ].astype(int)
        )
        if "multiplicity" in df_reference[dihedral_type_name].columns:
            df_reference[dihedral_type_name][["multiplicity"]] = (
                df_reference[dihedral_type_name][
                    ["multiplicity"]
                ].astype(int)
            )

    for dihedral_type in multibasin_root.findall(
        f".//dihedrals/dihedrals_type"
    ):
        dihedral_type_name = dihedral_type.get("name")

        if (
            "multiplicity"
            in df_reference.get(
                dihedral_type_name, pd.DataFrame()
            ).columns
            and "multiplicity"
            in df_additional.get(
                dihedral_type_name, pd.DataFrame()
            ).columns
        ):
            merge_on = ["i", "j", "k", "l", "multiplicity"]
        else:
            merge_on = ["i", "j", "k", "l"]

        # Merge the two DataFrames on i, j, k, l
        merged_dihedrals = pd.merge(
            df_reference[dihedral_type_name],
            df_additional[dihedral_type_name],
            on=merge_on,
            how="outer",
            suffixes=("_1", "_2"),
            indicator="source",
        )

        merged_dihedrals["weight"] = np.nanmin(
            merged_dihedrals[["weight_1", "weight_2"]],
            axis=1,
        )
        print(
            "\n\tWarning: for AA models, only `dihedral_cosine` are "
            "being re-weighted. The other dihedrals will use the `min` "
            "weight. Will need to adjust that for planar dihedrals later",
            flush=True,
        )

        merged_dihedrals.loc[
            merged_dihedrals["source"] == "left_only", "theta0_2"
        ] = merged_dihedrals.loc[
            merged_dihedrals["source"] == "left_only", "theta0_1"
        ]
        merged_dihedrals.loc[
            merged_dihedrals["source"] == "right_only", "theta0_1"
        ] = merged_dihedrals.loc[
            merged_dihedrals["source"] == "right_only", "theta0_2"
        ]

        merged_dihedrals["theta0"] = ff._angular_midpoint(
            merged_dihedrals["theta0_1"].to_numpy(),
            merged_dihedrals["theta0_2"].to_numpy(),
        )

        # 1) Build a new list of just the non-interaction children
        edited_dihedrals = [
            element
            for element in dihedral_type
            if element.tag != "interaction"
        ]

        # 2) Append your merged interactions
        for row in merged_dihedrals.itertuples(index=False):
            # Only change the angles that are in the atom_set
            if (row.i - 1) in atom_set:
                theta0 = row.theta0
            else:
                theta0 = row.theta0_1
                
            element = ET.Element(
                "interaction",
                attrib={
                    "i": str(row.i),
                    "j": str(row.j),
                    "k": str(row.k),
                    "l": str(row.l),
                    "theta0": f"{theta0:.5e}",
                    "weight": (
                        f"{row.weight:.5e}"
                        if row.weight != 1
                        else "1"
                    ),
                },
            )
            if "multiplicity" in merge_on:
                element.set("multiplicity", str(row.multiplicity))

            edited_dihedrals.append(element)

        # 3) In one go, reassign the children of dihedral_type
        dihedral_type[:] = edited_dihedrals

    return multibasin_xml


In [19]:
# this function captures the unsymmetric indices in the contact list, 
# represented with the first spoke.
from collections import Counter
def _get_unsymmetric_indices(df_contacts, i = 'i', j = 'j'):
    sorted_index_remainder = np.sort(df_contacts[[i, j]].values%N_subunit_CA, axis = 1)
    counts_remainder = Counter(map(tuple, sorted_index_remainder))
    unsymmetric = [item for item, count in counts_remainder.items() if count != 8]
    return unsymmetric

def process_contacts(
    reference_xml: ET.ElementTree,
    reference_pdb: pd.DataFrame,
    additional_xml: ET.ElementTree,
    additional_pdb: pd.DataFrame,
    multibasin_xml: ET.ElementTree,
    idx_from_additional_to_reference: dict[int, int],
    atom_set: set[int],
    scale_distances=1.0,
    iso_energy_threshold = 0.5,
) -> ET.ElementTree:
    """
    Processes contacts from the reference and additional XML files,
    converting them to the reference indices and merging them.
    The final contacts are averages of the parameters in both structures.
    """

    mode = "CG"

    if mode == "CG":
        epsilon = 1.0
        POW_ATTRACTION = 10
        COEFF_ATTRACTION = 6
        POW_REPULSION = 12
        COEFF_REPULSION = 5
    elif mode == "AA":
        epsilon = 1.0  # will be adjusted later
        POW_ATTRACTION = 6
        COEFF_ATTRACTION = 2
        POW_REPULSION = 12
        COEFF_REPULSION = 1

    else:
        raise ValueError(
            f"Unknown mode for contacts: {mode}. "
            "Supported modes are 'CG' and 'AA'."
        )

    def contact_energy(r, sigma):
        return epsilon * (
            COEFF_REPULSION * (sigma / r) ** POW_REPULSION
            - COEFF_ATTRACTION * (sigma / r) ** POW_ATTRACTION
        )

    def isoenergetic_distance(r1, r2):
        return (
            COEFF_ATTRACTION
            / COEFF_REPULSION
            * (
                (1 / r1**POW_ATTRACTION - 1 / r2**POW_ATTRACTION)
                / (1 / r1**POW_REPULSION - 1 / r2**POW_REPULSION)
            )
        ) ** (1 / (POW_REPULSION - POW_ATTRACTION))

    reference_root = reference_xml.getroot()
    if reference_root is None:
        raise ValueError(
            "The reference XML tree is empty or malformed."
        )
    additional_root = additional_xml.getroot()
    if additional_root is None:
        raise ValueError(
            "The additional XML tree is empty or malformed."
        )
    multibasin_root = multibasin_xml.getroot()
    if multibasin_root is None:
        raise ValueError(
            "The multibasin XML tree is empty or malformed."
        )

    contacts_converted_to_reference = {
        "i": [],
        "j": [],
        "A": [],
        "B": [],
    }

    for contact_type in additional_root.findall(
        ".//contacts/contacts_type"
    ):
        for interaction in contact_type.findall("interaction"):
            # Convert contact indices from additional to reference

            ii = int(interaction.attrib["i"])
            jj = int(interaction.attrib["j"])

            i = idx_from_additional_to_reference.get(ii)
            j = idx_from_additional_to_reference.get(jj)

            # when going thru additional, there's no option of
            # sigma_reference not existing if sigma_additional exists
            if i is not None and j is not None:
                contacts_converted_to_reference["i"].append(min(i, j))
                contacts_converted_to_reference["j"].append(max(i, j))

                contacts_converted_to_reference["A"].append(
                    interaction.attrib["A"]
                )
                contacts_converted_to_reference["B"].append(
                    interaction.attrib["B"]
                )

    i_list = contacts_converted_to_reference["i"]
    j_list = contacts_converted_to_reference["j"]
    contacts_converted_to_reference["sigma_reference"] = (
        np.linalg.norm(
            reference_pdb.loc[i_list, ["x", "y", "z"]].values
            - reference_pdb.loc[j_list, ["x", "y", "z"]].values,
            axis=1,
        )
        * 0.1  # Convert to nm
    ) * scale_distances

    df_additional = pd.DataFrame(
        data=contacts_converted_to_reference
    )
    df_additional[["A", "B"]] = df_additional[["A", "B"]].astype(
        float
    )
    df_additional[["i", "j"]] = df_additional[["i", "j"]].astype(int)
    df_additional["sigma_additional"] = (
        df_additional["A"]
        / df_additional["B"]
        * COEFF_ATTRACTION
        / COEFF_REPULSION
    ) ** (1 / (POW_REPULSION - POW_ATTRACTION))

    df_additional_in_set =\
        df_additional.loc[
            ((df_additional['i'] - 1).isin(atom_set) | (df_additional['j'] - 1).isin(atom_set))
        ].copy()
    df_additional_in_set.drop(["A", "B"], axis=1, inplace=True)
    # df_additional.drop(["A", "B"], axis=1, inplace=True)

    contacts_in_reference = {
        "i": [],
        "j": [],
        "A": [],
        "B": [],
    }

    idx_from_reference_to_additional = {
        v: k for k, v in idx_from_additional_to_reference.items()
    }

    for contacts_type in reference_root.findall(
        ".//contacts/contacts_type"
    ):
        for interaction in contacts_type.findall("interaction"):

            i = int(interaction.attrib["i"])
            j = int(interaction.attrib["j"])

            contacts_in_reference["i"].append(min(i, j))
            contacts_in_reference["j"].append(max(i, j))
            contacts_in_reference["A"].append(interaction.attrib["A"])
            contacts_in_reference["B"].append(interaction.attrib["B"])

    # convert index to float to allow NaN and add NaN row
    nan_row = pd.DataFrame([{}], index=[np.nan])
    float_indexed_additional_pdb = pd.concat([additional_pdb, nan_row])

    ii_list = np.array(
        [
            idx_from_reference_to_additional.get(i)
            for i in contacts_in_reference["i"]
        ],
        dtype=float,
    )
    jj_list = np.array(
        [
            idx_from_reference_to_additional.get(j)
            for j in contacts_in_reference["j"]
        ],
        dtype=float,
    )

    # this will give NaN for the contacts that are not in the additional PDB
    contacts_in_reference["sigma_additional"] = (
        np.linalg.norm(
            float_indexed_additional_pdb.loc[ii_list, ["x", "y", "z"]].values
            - float_indexed_additional_pdb.loc[jj_list, ["x", "y", "z"]].values,
            axis=1,
        )
        * 0.1  # Convert to nm
    ) * scale_distances

    df_reference = pd.DataFrame(data=contacts_in_reference)
    df_reference[["A", "B"]] = df_reference[["A", "B"]].astype(float)
    df_reference[["i", "j"]] = df_reference[["i", "j"]].astype(int)
    df_reference["sigma_reference"] = (
        df_reference["A"]
        / df_reference["B"]
        * COEFF_ATTRACTION
        / COEFF_REPULSION
    ) ** (1 / (POW_REPULSION - POW_ATTRACTION))

    df_reference["epsilon"] = (
        df_reference["A"]
        / COEFF_REPULSION
        * df_reference["sigma_reference"] ** -POW_REPULSION
    )

    if mode == "CG" and not np.isclose(
        np.nanmean(df_reference["epsilon"]), epsilon, atol=0.01
    ):
        print(
            "Warning! Contacts do not have the expected epsilon value for a CG model."
            f"Expected: {epsilon}, Found: {np.nanmean(df_reference['epsilon'])}."
        )

    # Only change the contacts that are in the atom_set, the rest will be the same as the reference
    df_reference_not_in_set =\
        df_reference.loc[
            ~((df_reference['i'] - 1).isin(atom_set) | (df_reference['j'] - 1).isin(atom_set))
        ].copy()
    df_reference_in_set =\
        df_reference.loc[
            ((df_reference['i'] - 1).isin(atom_set) | (df_reference['j'] - 1).isin(atom_set))
        ].copy()
    df_reference_in_set.drop(["A", "B"], axis=1, inplace=True)


    # Merge the two DataFrames on i, j
    merged_contacts = pd.merge(
        df_reference_in_set,
        df_additional_in_set,
        on=["i", "j"],
        how="outer",
        suffixes=("_1", "_2"),
        indicator="source",
    )  # this identifies common contact pairs

    if (
        np.mean(
            np.isclose(
                merged_contacts.loc[
                    merged_contacts.source == "both",
                    "sigma_reference_1",
                ],
                merged_contacts.loc[
                    merged_contacts.source == "both",
                    "sigma_reference_2",
                ],
                atol=0.01,
            )
        )
        == 1.0
    ):
        # prioritize the distance from the PDB
        merged_contacts["sigma_reference"] = merged_contacts[
            "sigma_reference_2"
        ].combine_first(merged_contacts["sigma_reference_1"])

    else:
        raise ValueError(
            "Inconsistent values of sigma_reference found! "
            "This indicates that the distances inferred from the contact interactions in the "
            "reference XML are not consistent with the distances from the reference PDB. "
            "Please check the input files."
        )

    if (
        np.mean(
            np.isclose(
                merged_contacts.loc[
                    (
                        (merged_contacts.source == "both")
                        & merged_contacts["sigma_additional_1"].notna()
                        & merged_contacts["sigma_additional_2"].notna()
                    ),
                    "sigma_additional_1",
                ],
                merged_contacts.loc[
                    (
                        (merged_contacts.source == "both")
                        & merged_contacts["sigma_additional_1"].notna()
                        & merged_contacts["sigma_additional_2"].notna()
                    ),
                    "sigma_additional_2",
                ],
                atol=0.01,
            )
        )
        == 1.0
    ):
        # prioritize the distance from the PDB
        merged_contacts["sigma_additional"] = merged_contacts[
            "sigma_additional_1"
        ].combine_first(merged_contacts["sigma_additional_2"])

    else:
        raise ValueError(
            "Inconsistent values of sigma_additional found! "
            "This indicates that the distances inferred from the contact interactions in the "
            "additional XML are not consistent with the distances from the additional PDB. "
            "Please check the input files."
        )

    assert (
        merged_contacts["sigma_reference"].notna().all()
    ), "NaN found in sigma_reference."

    # reminder: we can have NaN in sigma_additional for the contacts
    # that are not in the additional PDB

    # common contacts

    equal_distances = (
        merged_contacts["sigma_reference"] == merged_contacts["sigma_additional"]
    )

    merged_contacts.loc[equal_distances, "sigma_iso"] = merged_contacts.loc[
        equal_distances, "sigma_reference"
    ]
    merged_contacts.loc[~equal_distances, "sigma_iso"] = isoenergetic_distance(
        merged_contacts.loc[~equal_distances, "sigma_reference"],
        merged_contacts.loc[~equal_distances, "sigma_additional"],
    )

    # clean NaN for the contacts that are not in the additional PDB
    merged_contacts["sigma_iso"] = (
        merged_contacts["sigma_iso"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(merged_contacts["sigma_reference"])
    )

    merged_contacts["epsilon_iso"] = np.abs(
        contact_energy(
            r=merged_contacts["sigma_reference"],
            sigma=merged_contacts["sigma_iso"],
        )
    )

    ## for the cases where the eps_iso/eps > iso_energy_threshold
    # fix assignments for the cases where contacts are called in only ones
    # of the structures, but the isoenergetic distance indicates that the
    # contact is a common contact
    merged_contacts.loc[
        (merged_contacts["epsilon_iso"] > iso_energy_threshold * epsilon)
        & (merged_contacts["sigma_additional"].notna()),
        "source",
    ] = "both"

    ## for the cases where the eps_iso/eps <= iso_energy_threshold
    # when only one of the contacts is called by Shadow, go back to that one
    non_unique_called_in_left = (merged_contacts["epsilon_iso"] <= iso_energy_threshold * epsilon) & (
        merged_contacts["source"] == "left_only"
    )
    merged_contacts.loc[
        non_unique_called_in_left,
        "sigma_iso",
    ] = merged_contacts.loc[
        non_unique_called_in_left,
        "sigma_reference",
    ].values

    non_unique_called_in_right = (merged_contacts["epsilon_iso"] <= iso_energy_threshold * epsilon) & (
        merged_contacts["source"] == "right_only"
    )
    merged_contacts.loc[
        non_unique_called_in_right,
        "sigma_iso",
    ] = merged_contacts.loc[
        non_unique_called_in_right,
        "sigma_additional",
    ].values

    # when both contacts are called by Shadow, go back to the lower distance
    non_unique_called_in_both = (merged_contacts["epsilon_iso"] <= iso_energy_threshold * epsilon) & (
        merged_contacts["source"] == "both"
    )

    left_distance = merged_contacts.loc[non_unique_called_in_both, "sigma_reference"]
    right_distance = merged_contacts.loc[non_unique_called_in_both, "sigma_additional"]
    fix_non_unique_called_in_both = left_distance <= right_distance

    merged_contacts.loc[non_unique_called_in_both, "sigma_iso"] = left_distance.where(
        fix_non_unique_called_in_both, right_distance
    )

    merged_contacts.loc[non_unique_called_in_both, "source"] = np.where(
        fix_non_unique_called_in_both, "left_only", "right_only"
    )

    if mode == "AA":
        print(
            "\n\tAdjusting epsilon_c and dihedral weights for AA model...",
            end=" ",
            flush=True,
        )

        # adjust epsilon and the weight of the dihedrals
        num_common_contacts = len(
            merged_contacts.loc[merged_contacts["source"] == "both"]
        )

        dihedrals, df_dihedrals = _count_dihedrals(
            reference_pdb, multibasin_xml
        )
        num_backbone_dihedrals = (
            dihedrals["protein_bb"] + dihedrals["nucleic_bb"]
        )

        Nc = num_common_contacts
        Nbb = num_backbone_dihedrals
        Nscp = dihedrals["protein_sc"]
        Nscn = dihedrals["nucleic_sc"]

        N = len(reference_pdb)

        # variables: [Ec, Ebb, Escp, Escn]
        A = [
            [0, 1, 0, -1],
            [0, 1, -2, 0],
            [Nc, 2 * Nbb, -2 * Nscp, -2 * Nscn],
            [Nc, Nbb, Nscp, Nscn],
        ]
        b = [0, 0, 0, N]
        solution = np.linalg.solve(A, b)
        epsilon, weight_bb, weight_scp, weight_scn = solution

        df_dihedrals["epsilon"] = weight_bb
        df_dihedrals.loc[
            (df_dihedrals["molecule_type"] == "protein")
            & (df_dihedrals["dihedral_type"] == "sc"),
            "epsilon",
        ] = weight_scp
        df_dihedrals.loc[
            (df_dihedrals["molecule_type"] == "nucleic")
            & (df_dihedrals["dihedral_type"] == "sc"),
            "epsilon",
        ] = weight_scn

        multibasin_xml = _update_dihedral_weights(
            multibasin_xml,
            df_dihedrals,
        )
        print("Done.", flush=True)

    # before preparing for the flat bottom
    # make sure the unique contacts are symmetric
    unsymmetric = _get_unsymmetric_indices(merged_contacts.query('source == "both"'))  
    update_lookup = dict()
    for pair in unsymmetric:
        sel = np.all(np.sort(merged_contacts[['i', 'j']].values%N_subunit_CA, axis = 1) == pair, axis = 1)
        merged_contacts.loc[sel, 'source'] = 'both'

    # prepare for the flat bottom
    merged_contacts['r1'] = merged_contacts[['sigma_reference', 'sigma_additional']].min(axis=1)
    merged_contacts['r2'] = merged_contacts[['sigma_reference', 'sigma_additional']].max(axis=1)
    merged_contacts['A1'] = epsilon * COEFF_REPULSION * merged_contacts['r1'] ** POW_REPULSION
    merged_contacts['B1'] = epsilon * COEFF_ATTRACTION * merged_contacts['r1'] ** POW_ATTRACTION
    merged_contacts['A2'] = epsilon * COEFF_REPULSION * merged_contacts['r2'] ** POW_REPULSION
    merged_contacts['B2'] = epsilon * COEFF_ATTRACTION * merged_contacts['r2'] ** POW_ATTRACTION

    # replace new contacts in the mutlibasin_xml
    try:
        multibasin_contacts_root = multibasin_root.find("contacts")
        multibasin_contacts_root[:] = [] # Clear the existing contacts
    except:
        multibasin_contacts_root = ET.SubElement(multibasin_root, "contacts")

    # 10-12 potential 
    contacts_type_1 = ET.SubElement(multibasin_contacts_root, 'contacts_type', {'name': 'contact_1-10-12'}) 
    ET.SubElement(contacts_type_1, "expression", attrib={"expr": "A/r^12-B/r^10"})        
    ET.SubElement(contacts_type_1, "parameter").text = "A"
    ET.SubElement(contacts_type_1, "parameter").text = "B"


    # flat bottom potential 
    contacts_type_2 = ET.SubElement(multibasin_contacts_root, 'contacts_type', {'name': 'contact_2-flat-bottom'})
    ET.SubElement(contacts_type_2, "expression", attrib={"expr": "(1+A1/r^12-B1/r^10) * step(r1-r) + (1+A2/r^12-B2/r^10) * step(r-r2)"})      
    ET.SubElement(contacts_type_2, "parameter").text = "A1"
    ET.SubElement(contacts_type_2, "parameter").text = "B1"
    ET.SubElement(contacts_type_2, "parameter").text = "A2"
    ET.SubElement(contacts_type_2, "parameter").text = "B2"
    ET.SubElement(contacts_type_2, "parameter").text = "r1"
    ET.SubElement(contacts_type_2, "parameter").text = "r2"


    # write the contacts not in the set
    for row in df_reference_not_in_set.itertuples(index=False):
        ET.SubElement(
            contacts_type_1, "interaction",
            attrib = {
                "i":str(row.i), "j":str(row.j), 
                "A":f"{row.A:.5e}", "B":f"{row.B:.5e}"
            },
        )

    # write the common contacts in the set
    for row in merged_contacts.itertuples(index=False):
        if row.source == 'both':
            ET.SubElement(
                contacts_type_2, "interaction",
                attrib = {
                    "i":str(row.i), "j":str(row.j), 
                    "A1":f"{row.A1:.5e}", "B1":f"{row.B1:.5e}",
                    "A2":f"{row.A2:.5e}", "B2":f"{row.B2:.5e}",
                    "r1":f"{row.r1:.5e}", "r2":f"{row.r2:.5e}"
                },
            )
    
    return multibasin_xml, merged_contacts


In [20]:
def remove_unstable_dihedrals(
    xml: ET.ElementTree,
    top: dict[str, pd.DataFrame] | None = None,
) -> ET.ElementTree:
    """
    Remove dihedral interactions that may cause numerical instability.
    When the angles in a dihedral are close to 0 or 180 degrees,
    the dihedral potential can become very steep, leading to
    numerical instability during simulations.

    Args:
        xml (ET.ElementTree): The XML tree containing dihedral information.
        top (dict[str, pd.DataFrame] | None): The topology dictionary containing the angle information, if that is not defined in the xml.
    Returns:
        ET.ElementTree: The modified XML tree with specified dihedrals removed.
    """

    def _strlist2tuple(*strlist: str) -> tuple:
        return tuple(sorted(map(int, strlist)))

    edited_xml = copy.deepcopy(xml)
    root = edited_xml.getroot()

    idx_in_faulty_angles = set()

    for angle_type in root.find("angles"):
        if angle_type.get("name") == 'angle_flat_bottom':
            for interaction in angle_type.findall("interaction"):
                if (
                    float(interaction.attrib["theta1"]) <= np.deg2rad(30)
                    or float(interaction.attrib["theta1"])
                    >= np.deg2rad(150)
                ) or (
                    float(interaction.attrib["theta2"]) <= np.deg2rad(30)
                    or float(interaction.attrib["theta2"])
                    >= np.deg2rad(150)
                ):
                    idx_in_faulty_angles.add(
                        _strlist2tuple(
                            interaction.attrib["i"],
                            interaction.attrib["j"],
                            interaction.attrib["k"],
                        )
                    )
        elif angle_type.get("name") == 'angle_harmonic':
            for interaction in angle_type.findall("interaction"):
                if (
                    float(interaction.attrib["theta0"]) <= np.deg2rad(30)
                    or float(interaction.attrib["theta0"])
                    >= np.deg2rad(150)
                ):
                    idx_in_faulty_angles.add(
                        _strlist2tuple(
                            interaction.attrib["i"],
                            interaction.attrib["j"],
                            interaction.attrib["k"],
                        )
                    )
        else:
            raise ValueError(
                f"Unknown angle type: {angle_type.get('name')}. "
                "Supported angle types are 'angle_flat_bottom' and 'angle_harmonic'."
            )
        
    if (
        top is not None
        and "angles" in top
        and "th0(deg)" in top["angles"].columns
    ):
        top["angles"].rename(columns={"th0(deg)": "th0_deg"}, inplace=True) # inplace to save memory
        for row in top["angles"].itertuples(index=False):
            if (row.th0_deg <= 30) or (row.th0_deg >= 150):
                idx_in_faulty_angles.add(
                    _strlist2tuple(
                        row.ai,
                        row.aj,
                        row.ak,
                    )
                )
        top["angles"].rename(columns={"th0_deg": "th0(deg)"}, inplace=True)

    for dihedral_type in root.find("dihedrals"):
        modified_dihedrals = []
        for element in dihedral_type:
            if element.tag == "interaction":
                dihedral_idx = set()

                dihedral_idx.add(
                    _strlist2tuple(
                        element.attrib["j"],
                        element.attrib["k"],
                        element.attrib["l"],
                    )
                )
                dihedral_idx.add(
                    _strlist2tuple(
                        element.attrib["i"],
                        element.attrib["k"],
                        element.attrib["l"],
                    )
                )
                dihedral_idx.add(
                    _strlist2tuple(
                        element.attrib["i"],
                        element.attrib["j"],
                        element.attrib["l"],
                    )
                )
                dihedral_idx.add(
                    _strlist2tuple(
                        element.attrib["i"],
                        element.attrib["j"],
                        element.attrib["k"],
                    )
                )

                if (
                    len(
                        dihedral_idx.intersection(
                            idx_in_faulty_angles
                        )
                    )
                    > 0
                ):
                    continue
                else:
                    modified_dihedrals.append(element)
            else:
                modified_dihedrals.append(element)
        dihedral_type[:] = modified_dihedrals

    return edited_xml

In [21]:
def define_dualbasin_SBM_subegion(
    reference_top: dict[str, pd.DataFrame],
    reference_xml: ET.ElementTree,
    reference_pdb: pd.DataFrame,
    additional_top: dict[str, pd.DataFrame],
    additional_xml: ET.ElementTree,
    additional_pdb: pd.DataFrame,
    idx_from_additional_to_reference: dict[int, int],
    atom_set: set[int],
    xml_file: str = "smog.xml",
    top_file: str = "smog.top",
) -> tuple[
    dict[str, pd.DataFrame], ET.ElementTree, pd.DataFrame
]:
    multibasin_top = copy.deepcopy(reference_top)
    multibasin_xml = copy.deepcopy(reference_xml)  

    # For the bonds
    print("Processing bonds...", flush=True, end="")
    multibasin_top, multibasin_xml = process_bonds(
        reference_top,
        additional_top,
        multibasin_top,
        idx_from_additional_to_reference,
        multibasin_xml,
        atom_set,
    )
    print(" Done.", flush=True)

    # For the angles

    print("Processing angles...", flush=True, end="")
    multibasin_top, multibasin_xml = process_angles(
        reference_top,
        additional_top,
        multibasin_top,
        idx_from_additional_to_reference,
        multibasin_xml,
        atom_set,
    )
    print(" Done.", flush=True)

    # For the dihedrals

    print("Processing dihedrals...", flush=True, end="")
    multibasin_xml = process_dihedrals(
        reference_xml,
        additional_xml,
        multibasin_xml,
        idx_from_additional_to_reference,
        atom_set,
    )
    print(" Done.", flush=True)

    # For the contacts

    print("Processing contacts...", flush=True, end="")
    multibasin_xml, _ = process_contacts(
        reference_xml,
        reference_pdb,
        additional_xml,
        additional_pdb,
        multibasin_xml,
        idx_from_additional_to_reference,
        atom_set,
    )

    multibasin_top = ff._update_exclusions(
        xml=multibasin_xml,
        top=multibasin_top,
    )
    print(" Done.", flush=True)

    multibasin_xml = remove_unstable_dihedrals(multibasin_xml)

    print("Saving files...", flush=True, end="")

    ff.save_top(multibasin_top, top_file)
    ff.save_xml(multibasin_xml, xml_file)

    # df_contacts.to_pickle(contact_file)

    print(" Done.", flush=True)

    return multibasin_top, multibasin_xml

# Bridge dual basin

In [22]:
define_dualbasin_SBM_subegion(
    reference_top = top_c,
    reference_xml = xml_c,
    reference_pdb = df_c,
    additional_top = top_d,
    additional_xml = xml_d,
    additional_pdb = df_d,
    idx_from_additional_to_reference = dict_idx,
    atom_set = ring_all_indices[-1], # Bridge
    xml_file = "constricted_CA_bridge_dualbasin.xml",
    top_file = "constricted_CA_bridge_dualbasin.top",
)

Processing bonds... Done.
Processing angles... Done.
Processing dihedrals...
 Done.
Processing contacts... Done.
Saving files... Done.


({'defaults':   nbfunc comb-rule gen-pairs fudgeLJ fudgeQQ
  0      1         1        no       1       1,
  'atomtypes':    name    mass    charge ptype           c6          c12
  0  NB_1  1.0000  0.000000     A  0.00000e+00  1.67772e-05,
  'moleculetype':             name nrexcl
  0  Macromolecule      3,
  'atoms':             nr  type   resnr residue atom    cgnr
  0            1  NB_1       1     SER   CA       1
  1            2  NB_1       2     LYS   CA       2
  2            3  NB_1       3     ALA   CA       3
  3            4  NB_1       4     ASP   CA       4
  4            5  NB_1       5     VAL   CA       5
  ...        ...   ...     ...     ...  ...     ...
  624235  624236  NB_1  624236     ASN   CA  624236
  624236  624237  NB_1  624237     HIS   CA  624237
  624237  624238  NB_1  624238     VAL   CA  624238
  624238  624239  NB_1  624239     ASN   CA  624239
  624239  624240  NB_1  624240    PHET   CA  624240
  
  [624240 rows x 6 columns],
  'bonds':             ai